# 05 — Async CTRS-R Evaluation (Single Model)

Evaluate **one model at a time** using simulated CBT sessions scored with the CTRS-R rubric.

Set `MODEL_NAME` below to either:
- A folder name in `$SSD_ROOT/models/` (e.g. `"Qwen3.5-4B-SFT-iter_01"`) for a fine-tuned model
- `"base"` to evaluate the base model directly

Results are saved to `reports/<MODEL_NAME>/`.

In [11]:
from dotenv import load_dotenv
load_dotenv("../.env")

import asyncio
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from openai import AsyncOpenAI
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : NVIDIA RTX 6000 Ada Generation
VRAM    : 50.9 GB


## Configuration

Set `MODEL_NAME` to the model you want to evaluate:
- `"base"` → evaluates the base Qwen model directly
- Any other string → looks for that folder in `$SSD_ROOT/models/`

In [12]:
# ── Model to evaluate ─────────────────────────────────────────────────────────
# Set this to "base" for the base model, or a model folder name in $SSD_ROOT/models/
# Self-play final model = last CACTUS self-play iteration (iter_02).
MODEL_NAME = "Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged"

# ── Base model (used for loading adapters or as the eval target when MODEL_NAME="base")
BASE_MODEL_NAME = "Qwen/Qwen3.5-4B"

# ── Paths ─────────────────────────────────────────────────────────────────────
SSD_ROOT = Path(os.environ.get("SSD_ROOT", "/tmp"))

if MODEL_NAME == "base":
    MODEL_PATH = BASE_MODEL_NAME  # HuggingFace model ID
else:
    MODEL_PATH = SSD_ROOT / "models" / MODEL_NAME

# ── DeepSeek patient / judge / moderator model ───────────────────────────────
PATIENT_MODEL    = "deepseek-v4-flash"
JUDGE_MODEL      = "deepseek-v4-flash"
MODERATOR_MODEL  = "deepseek-v4-flash"
MIN_TURNS        = 5     # minimum exchanges before moderator can end session

# ── Conversation ─────────────────────────────────────────────────────────────
MAX_TURNS             = 20    # hard cap on therapist-patient exchanges
THERAPIST_MAX_TOKENS  = 256   # match src/alignment/self_play.py
PATIENT_MAX_TOKENS    = 256
JUDGE_MAX_TOKENS      = 50000
THERAPIST_BATCH_SIZE  = 8     # GPU batch size for therapist inference

# ── Decoding guards (match src/alignment/self_play.py) ───────────────────────
NO_REPEAT_NGRAM_SIZE = 0   # 0 = disabled (testing); if enabling, prefer 5 over 3
REPETITION_PENALTY   = 1.1
PATIENT_FREQUENCY_PENALTY = 0.5
THERAPIST_MAX_PROMPT_TOKENS = 16384  # match self_play.py; left-truncate over-long prompts

# ── Concurrency ──────────────────────────────────────────────────────────────
API_CONCURRENCY = 500   # semaphore cap (DeepSeek allows ~2500)

# ── Output ────────────────────────────────────────────────────────────────────
REPORTS_DIR = Path("../reports") / MODEL_NAME
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_JSONL  = REPORTS_DIR / "ctsr_eval_transcripts.jsonl"
EVAL_CSV    = REPORTS_DIR / "ctsr_eval_scores.csv"

# ── Therapist system prompt ──────────────────────────────────────────────────
THERAPIST_SYSTEM = (
    "You are a warm, highly skilled CBT (Cognitive Behavioural Therapy) therapist in a one-to-one "
    "session with a client. Conduct the session as a real therapist would, following CBT structure "
    "and staying fully in role at all times.\n\n"
    "SESSION STRUCTURE — progress through these phases over the course of the session; never get "
    "stuck in one phase:\n"
    "1. Open with a brief check-in: a mood check, acknowledge anything that has changed since last "
    "time, and collaboratively set an agenda for what to focus on today.\n"
    "2. Explore the presenting concern with Socratic questioning — draw out the specific thoughts, "
    "emotions, and behaviours and the situations that trigger them.\n"
    "3. Help the client see the links between their thoughts, feelings, and behaviours, and gently "
    "test unhelpful or distorted thinking against the evidence.\n"
    "4. Work collaboratively toward a shared understanding and a concrete, practical coping "
    "strategy or intervention.\n"
    "5. Toward the end, agree a specific between-session homework/action plan, invite the client's "
    "feedback on the session, and close warmly.\n\n"
    "STYLE:\n"
    "- Be warm, empathic, genuine, non-judgmental, and professionally boundaried.\n"
    "- Validate emotions before gently challenging thoughts.\n"
    "- Use plain, everyday language — never clinical jargon or the names of techniques.\n"
    "- Keep every turn short and focused: 2-4 sentences, asking at most one or two questions.\n\n"
    "CRITICAL RULES — follow these on every single turn, without exception:\n"
    "- NEVER repeat yourself. Do not reuse a sentence, question, or phrasing you have already used "
    "earlier in this conversation. Every turn must contain genuinely new content.\n"
    "- Do NOT re-ask anything the client has already answered. Read what they just said, "
    "acknowledge it specifically, and build on it.\n"
    "- Always move the session forward. If a thread is resolved or the client starts repeating "
    "themselves, advance to the next phase rather than circling back to the same point.\n"
    "- Vary your openings and wording; avoid formulaic stock phrases.\n"
    "- Write ONLY the therapist's spoken reply, as natural prose. No stage directions, no "
    "parentheticals, no inner monologue, no lists, no headings, and no meta-commentary.\n"
    "- Respond in English only."
)

print(f"Model name : {MODEL_NAME}")
print(f"Model path : {MODEL_PATH}")
print(f"Patient    : {PATIENT_MODEL}")
print(f"Judge      : {JUDGE_MODEL}")
print(f"Concurrency: {API_CONCURRENCY}")
print(f"Batch size : {THERAPIST_BATCH_SIZE}")
print(f"Reports    : {REPORTS_DIR.resolve()}")

Model name : Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged
Model path : /data_1_8TB_ssd/kevint/models/Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged
Patient    : deepseek-v4-flash
Judge      : deepseek-v4-flash
Concurrency: 500
Batch size : 8
Reports    : /home/kevint/cse-reu/reports/Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged


## Clinical Vignettes — CACTUS held-out test split

In [13]:
# Held-out CACTUS test vignettes (see src/data/prepare_cactus.py)
CACTUS_TEST_PATH = Path("../data/processed/cactus_vignettes_test.jsonl")


def _patient_system_prompt(intake_form: str, patterns: str) -> str:
    """Patient agent persona — identical to src/alignment/self_play.py."""
    return (
        "You are role-playing as a real person attending a CBT (Cognitive Behavioural Therapy) "
        "session as the client. Fully embody the person described below — their background, life "
        "circumstances, and presenting problem are yours.\n\n"
        "=== YOUR PROFILE (intake form) ===\n"
        f"{intake_form}\n\n"
        "=== HOW YOU THINK ===\n"
        f"Your thinking tends to fall into these patterns: {patterns}. "
        "Let them naturally shape how you interpret events and respond, but NEVER name or describe "
        "them — you are not aware of them as 'distortions'; they simply feel true to you.\n\n"
        "=== HOW TO BEHAVE ===\n"
        "- Stay fully in character as this person for the entire conversation.\n"
        "- Speak naturally in everyday language; never use clinical or therapy jargon.\n"
        "- Reveal your problems gradually as the therapist builds rapport — don't dump everything at once.\n"
        "- Be a little guarded at first; let the therapist guide you toward insight rather than handing it over.\n"
        "- React authentically (resistance, relief, doubt, emotion) in a way that fits your profile.\n"
        "- Keep each reply to 2–5 sentences.\n"
        "- Never break character, never mention being an AI, and never reference therapy techniques by name."
    )


def person_to_vignette(p: dict) -> dict:
    """Build the eval vignette (name / background / patient system_prompt) from a CACTUS person."""
    patterns = ", ".join(p["patterns"])
    background = f"{p['intake_form']}\n\nCognitive distortion patterns: {patterns}"
    return {
        "name": p["id"],
        "background": background,
        "system_prompt": _patient_system_prompt(p["intake_form"], patterns),
    }


with open(CACTUS_TEST_PATH, encoding="utf-8") as f:
    _test_people = [json.loads(line) for line in f if line.strip()]

VIGNETTES = [person_to_vignette(p) for p in _test_people]

print(f"Loaded {len(VIGNETTES)} held-out CACTUS test vignettes")
for v in VIGNETTES[:3]:
    print(f"  {v['name']}: {v['background'][:80]}…")


Loaded 100 held-out CACTUS test vignettes
  cactus_00000: Name:
Jennifer Dakota
Age:
41
Gender:
female
Occupation: Not specified
Education…
  cactus_00001: Name:
Cory Adams
Age:
30
Gender:
male
Occupation: Not specified
Education: Not s…
  cactus_00002: Name:
Derek Mitchell
Age:
29
Gender:
male
Occupation: Not disclosed
Education: N…


## CTRS-R Rubric — 11 Items, 0–3 Scale

In [14]:
CTSR_ITEMS = [
    {
        "number": 1, "name": "Agenda", "key": "item_1_agenda", "short_label": "Agenda",
        "criteria": (
            "Did the therapist...\n"
            "• Provide transition to the previous session?\n"
            "• Identify significant events [positive and/or negative] since previous session?\n"
            "• Review Action Plan [complete review may be done as part of the agenda]?\n"
            "• Conduct a mood check?\n"
            "• Identify specific goals or problems to work on during the session?"
        ),
        "anchors": {
            0: "If therapist completed none of the above items",
            1: "If therapist completed one or more but not all of the above items",
            2: "If therapist completed all five of the above items",
            3: "If therapist completed all five of the above items PLUS… Made certain that all items important to the client were addressed and prioritized; Followed the agenda throughout the session unless there was an overt discussion about deviating from the agenda.",
        },
    },
    {
        "number": 2, "name": "Feedback", "key": "item_2_feedback", "short_label": "Feedback",
        "criteria": (
            "Did the therapist...\n"
            "• Ascertain the client's reaction to the session, the therapist, or the therapeutic process?\n"
            "• Ensure that the client understood and agreed with the treatment plan?\n"
            "• Respond appropriately to feedback?"
        ),
        "anchors": {
            0: "If therapist completed none of the above items",
            1: "If therapist completed one or more but not all of the above items",
            2: "If therapist completed all three of the above items",
            3: "If the therapist completed all three of the above items PLUS… The therapist fluidly requested feedback throughout the session [agenda, transitions, use of techniques, and/or developing an Action Plan].",
        },
    },
    {
        "number": 3, "name": "Understanding", "key": "item_3_understanding", "short_label": "Understanding",
        "criteria": (
            "Did the therapist...\n"
            "• Demonstrate they generally heard and understood the content of what the client expressed "
            "through repeating, summarizing, etc. what the client said during the session?"
        ),
        "anchors": {
            0: "If therapist did not demonstrate the above item",
            1: "If therapist inconsistently listened and reflected the client's statements",
            2: "If therapist listened and reflected the client's statements throughout the session",
            3: "If the therapist consistently listened and reflected throughout the session PLUS… Therapist demonstrated recognition of understanding the client's emotional state through acknowledgement, reflection, empathy; Discussed the client's emotional state within the context of the conceptualization; Demonstration of emotional state is accomplished by a combination of words, expressions, gestures, tone, and body language throughout the session.",
        },
    },
    {
        "number": 4, "name": "Interpersonal Effectiveness", "key": "item_4_interpersonal_effectiveness", "short_label": "Interpersonal\nEffect.",
        "criteria": (
            "Throughout the session, did the therapist…\n"
            "• Demonstrate concern for client and help the client reach their goals?\n"
            "• Provide positive reinforcement for actions taken by the client (e.g. completing action plans)?\n"
            "• Maintain professional and ethical behavior?"
        ),
        "anchors": {
            0: "If therapist completed none of the above items",
            1: "If therapist completed one or two, but not all three of the above items",
            2: "If therapist completed all three of the above items",
            3: "If the therapist completed all three of the above items PLUS… Through words, gestures, and expressions, demonstrated warmth, genuineness, and unconditional acceptance (absence of judgment) by making positive statements about the client's character or characteristics (e.g. strength, determination, caring, vision, values, integrity, etc.)",
        },
    },
    {
        "number": 5, "name": "Collaboration", "key": "item_5_collaboration", "short_label": "Collaboration",
        "criteria": (
            "Did the therapist...\n"
            "• Ask the client for input/agreement when setting the agenda and respond appropriately to the input?\n"
            "• Ask the client for input/agreement when selecting or using CBT techniques and respond appropriately to the input?\n"
            "• Ask the client for input/agreement when determining the Action Plan to be followed between sessions and responded appropriately to the input?"
        ),
        "anchors": {
            0: "If therapist completed none of the above items",
            1: "If therapist completed one or more but not all three of the above items",
            2: "If therapist completed all three of the above items",
            3: "If the therapist completed all three of the above items PLUS… Throughout the session, the therapist made a consistent effort to invite client's participation/agreement on every major decision about the session and responded appropriately. The collaboration resulted in a mutually agreeable direction for the session.",
        },
    },
    {
        "number": 6, "name": "Pacing and Efficient Use of Time", "key": "item_6_pacing", "short_label": "Pacing",
        "criteria": (
            "Did the therapist...\n"
            "• Allocate appropriate time for transition and agenda setting; intervention(s); feedback and action planning?\n"
            "• Complete the session within 40 – 60 minutes?"
        ),
        "anchors": {
            0: "If therapist completed none of the above items",
            1: "If therapist completed one but not both of the above items",
            2: "If therapist completed both of the above items",
            3: "If the therapist completed both of the above items PLUS… Provided pacing that allowed discussion to seamlessly move through each of the different segments; AND, if needed, made appropriate attempts to limit peripheral or unproductive discussion; AND the session was conducted within 45 – 55 minutes",
        },
    },
    {
        "number": 7, "name": "Guided Discovery", "key": "item_7_guided_discovery", "short_label": "Guided\nDiscovery",
        "criteria": (
            "Did the therapist...\n"
            "• Design and conduct the session to help the client achieve a cognitive shift regarding agenda items?\n"
            "• Throughout the session, avoid showing bias and avoid use of directions, arguments, or coercion to lead the client "
            "to 'see' things the way the therapist thinks the client should see them?\n"
            "• Assess the cognitive shift following an intervention?"
        ),
        "anchors": {
            0: "If the therapist made no attempt to help the client achieve a cognitive shift",
            1: "If the therapist completed one or more but not all of the above items",
            2: "If the therapist completed all three of the above items",
            3: "If the therapist completed all three of the above items PLUS… Throughout the session, skillfully utilized the process of discovery to help the client arrive at their own conclusions; Assess the potential impact of the cognitive shift on the client's emotions and behaviors.",
        },
    },
    {
        "number": 8, "name": "Focus on Key Cognitions and Behaviors", "key": "item_8_focus_cognitions_behaviors", "short_label": "Focus on Key\nCog. & Beh.",
        "criteria": (
            "Did the therapist...\n"
            "• Focus on specific cognitions, images, sensations, emotions, behaviors, and or meanings about aspirations "
            "or challenges associated with the sessions Agenda item(s)?"
        ),
        "anchors": {
            0: "If the therapist did not focus on any particular item during the session",
            1: "If the therapist focused on an issue that was unrelated to Agenda items or was unable to elicit specific cognitions, images, sensations, emotions, behaviors, and/or meanings related to Agenda items",
            2: "If the therapist focused on specific cognitions, images, sensations, emotions, behaviors, and/or meanings about aspirations or challenges associated with the sessions Agenda items",
            3: "If the therapist completed the above item PLUS… The items(s) were the most relevant cognitions, images, sensations, emotions, and/or meanings that held greatest promise for a positive impact on the client's aspirations or challenges related to the sessions agenda item(s).",
        },
    },
    {
        "number": 9, "name": "Strategy for Change", "key": "item_9_strategy_for_change", "short_label": "Strategy\nfor Change",
        "criteria": (
            "Did the therapist...\n"
            "• Discuss evidence-based (CBT) techniques as part of an overall strategy for change with the client?\n"
            "• Select and use at least one identifiable evidence-based technique that was appropriate for the agenda item being addressed?"
        ),
        "anchors": {
            0: "If the therapist did not appear to have any strategy that incorporated use of evidence-based (CBT) techniques",
            1: "If the therapist appeared to have a strategy that did not include use of an appropriate evidence-based (CBT) technique",
            2: "If the therapist discussed an overall strategy for change with the client and used at least one appropriate evidence-based (CBT) technique",
            3: "If the therapist completed both of the items above PLUS… The therapist explained the rationale for use of the technique; Offered other options (if applicable); Obtained the client's agreement to participate in use of the techniques.",
        },
    },
    {
        "number": 10, "name": "Application of CBT Technique", "key": "item_10_application_cbt_technique", "short_label": "Application\nof CBT Tech.",
        "criteria": (
            "Did the therapist...\n"
            "• Apply a CBT technique with sufficient skill that the technique was recognizable?\n"
            "• Apply a CBT technique in such a way that it would likely facilitate change in a motivated client?"
        ),
        "anchors": {
            0: "If the therapist attempts to apply a CBT technique was not done with sufficient skill that it was recognizable",
            1: "If the therapist achieved one of the above items but not the other one",
            2: "If the therapist performed the technique with sufficient skill that it accomplished both of the above items",
            3: "If the therapist accomplished both of the above items PLUS The therapist demonstrated good familiarity with the technique; The therapist was comfortable applying the technique; The therapist applied the technique in a technically correct manner (i.e. as the technique is described in the literature).",
        },
    },
    {
        "number": 11, "name": "Action Plan", "key": "item_11_action_plan", "short_label": "Action\nPlan",
        "criteria": (
            "Did the therapist...\n"
            "• Review the Action Plan from the previous session?\n"
            "• Ask the client to provide input/agreement or incorporate spontaneously offered ideas into the development of a new Action Plan?\n"
            "• Develop an Action Plan based on work done in the current session [and/or continued from a previous session, if applicable] "
            "that, if completed, the Action Plan would answer a question, or help the client to better cope, develop a new skill, or improve their relationships?"
        ),
        "anchors": {
            0: "If the therapist did not complete any of the above items",
            1: "If the therapist completed one or more but not all of the items listed above",
            2: "If the therapist completed all three of the items listed above",
            3: "If the therapist completed all of the items listed above PLUS… The therapist ensured that the client knew what to do, was capable of doing it, and it was specified when, where, how often, and how long to do the Action Plan; and The therapist assessed the reasonable likelihood that the client would complete the Action Plan; and The therapist addressed any challenges or obstacles that would potentially reduce the likelihood of the client completing the Action Plan.",
        },
    },
]

ITEM_KEYS   = [it["key"]         for it in CTSR_ITEMS]
ITEM_LABELS = [it["short_label"] for it in CTSR_ITEMS]

print(f"CTRS-R items loaded: {len(CTSR_ITEMS)}")
for it in CTSR_ITEMS:
    print(f"  {it['number']:>2}. {it['name']}")

CTRS-R items loaded: 11
   1. Agenda
   2. Feedback
   3. Understanding
   4. Interpersonal Effectiveness
   5. Collaboration
   6. Pacing and Efficient Use of Time
   7. Guided Discovery
   8. Focus on Key Cognitions and Behaviors
   9. Strategy for Change
  10. Application of CBT Technique
  11. Action Plan


## Async DeepSeek Client

In [15]:
ds_async = AsyncOpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

SEM = asyncio.Semaphore(API_CONCURRENCY)

print(f"Async DeepSeek client initialised (semaphore={API_CONCURRENCY}).")

Async DeepSeek client initialised (semaphore=500).


## Conversation Engine (Lockstep Async)

All 112 vignettes advance together, round by round:
1. **Therapist turns** — sequential on GPU (one `model.generate()` per active conversation)
2. **Patient turns** — all fired concurrently via `asyncio.gather()`
3. **Moderator checks** — all fired concurrently via `asyncio.gather()`
4. Finished conversations are removed from the active set

In [16]:
def strip_meta(text: str) -> str:
    """Remove leaked italic-parenthetical scratchpad, e.g. *(Self-Correction: ...)* / *(Wait, ...)*."""
    cleaned = re.sub(r"\*\(.*?\)\*", "", text, flags=re.DOTALL)
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()


def strip_thinking(text: str) -> str:
    """Strip Qwen3 <think>...</think> blocks and leaked inline scratchpad."""
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return strip_meta(cleaned)


def to_hf_messages(turns: list[dict], system_prompt: str) -> list[dict]:
    """Convert shared turn list to HF chat format (therapist POV)."""
    messages = [{"role": "system", "content": system_prompt}]
    for t in turns:
        role = "user" if t["role"] == "patient" else "assistant"
        messages.append({"role": role, "content": t["content"]})
    return messages


def to_patient_messages(turns: list[dict], system_prompt: str) -> list[dict]:
    """Convert shared turn list to chat format (patient POV)."""
    messages = [{"role": "system", "content": system_prompt}]
    for t in turns:
        role = "user" if t["role"] == "therapist" else "assistant"
        messages.append({"role": role, "content": t["content"]})
    return messages


# ── Batched therapist inference ───────────────────────────────────────────────

def _build_therapist_prompt(tokenizer, turns: list[dict], system_prompt: str) -> str:
    """Build one chat-template prompt for therapist inference."""
    messages = to_hf_messages(turns, system_prompt)
    if len(messages) == 1:
        messages.append({"role": "user", "content": "Please begin the session."})
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )


def _therapist_stop_ids(tokenizer) -> list[int]:
    """Return valid EOS/turn-end token ids for generation."""
    ids = [tokenizer.eos_token_id]
    try:
        ids.append(tokenizer.convert_tokens_to_ids("<|im_end|>"))
    except Exception:
        pass
    return sorted({int(x) for x in ids if isinstance(x, int) and x >= 0})


def therapist_turn_batch(
    model, tokenizer,
    turns_batch: list[list[dict]],
    system_prompts: list[str],
    batch_size: int = THERAPIST_BATCH_SIZE,
) -> list[str]:
    """Generate multiple therapist utterances in batched GPU calls."""
    if not turns_batch:
        return []

    prompt_texts = [
        _build_therapist_prompt(tokenizer, turns, sp)
        for turns, sp in zip(turns_batch, system_prompts)
    ]

    # Sort by length to reduce padding waste
    order = sorted(range(len(prompt_texts)), key=lambda i: len(prompt_texts[i]))
    responses: list[str | None] = [None] * len(prompt_texts)
    stop_ids = _therapist_stop_ids(tokenizer)

    old_padding_side = getattr(tokenizer, "padding_side", "right")
    old_truncation_side = getattr(tokenizer, "truncation_side", "right")
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    try:
        for start in range(0, len(order), batch_size):
            batch_indices = order[start:start + batch_size]
            batch_prompts = [prompt_texts[i] for i in batch_indices]

            inputs = tokenizer(
                batch_prompts, return_tensors="pt", padding=True,
                truncation=True, max_length=THERAPIST_MAX_PROMPT_TOKENS,
            ).to(model.device)

            with torch.inference_mode():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=THERAPIST_MAX_TOKENS,
                    do_sample=True, temperature=0.7, top_p=0.9, top_k=50,
                    repetition_penalty=REPETITION_PENALTY,
                    no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                    eos_token_id=stop_ids,
                    pad_token_id=tokenizer.pad_token_id,
                    use_cache=True,
                )

            gen_start = inputs["input_ids"].shape[1]
            new_token_ids = output_ids[:, gen_start:]
            decoded = tokenizer.batch_decode(new_token_ids, skip_special_tokens=True)

            for original_idx, raw in zip(batch_indices, decoded):
                responses[original_idx] = strip_thinking(raw)

            del inputs, output_ids, new_token_ids
            torch.cuda.empty_cache()
    finally:
        tokenizer.padding_side = old_padding_side
        tokenizer.truncation_side = old_truncation_side

    return [r if r is not None else "" for r in responses]


def append_batched_therapist_turns(
    model, tokenizer,
    all_turns: list[list[dict]],
    active: list[int],
    batch_size: int = THERAPIST_BATCH_SIZE,
) -> None:
    """Append one therapist turn to every active conversation using batched inference."""
    if not active:
        return
    turns_batch = [all_turns[idx] for idx in active]
    prompt_batch = [THERAPIST_SYSTEM] * len(active)
    responses = therapist_turn_batch(model, tokenizer, turns_batch, prompt_batch, batch_size)
    for idx, t_msg in zip(active, responses):
        all_turns[idx].append({"role": "therapist", "content": t_msg})


# ── Async API calls ───────────────────────────────────────────────────────────

async def patient_turn_async(client: AsyncOpenAI, vignette: dict, turns: list[dict]) -> str:
    """Generate patient's next utterance via DeepSeek (async)."""
    messages = to_patient_messages(turns, vignette["system_prompt"])
    async with SEM:
        resp = await client.chat.completions.create(
            model=PATIENT_MODEL,
            messages=messages,
            max_tokens=PATIENT_MAX_TOKENS,
            temperature=0.85,
            frequency_penalty=PATIENT_FREQUENCY_PENALTY,
            extra_body={"thinking": {"type": "disabled"}},
        )
    return strip_meta((resp.choices[0].message.content or "").strip())


async def should_end_session_async(client: AsyncOpenAI, turns: list[dict]) -> bool:
    """Moderator agent: checks if session should end (async)."""
    transcript_text = "\n".join(
        f"{'Therapist' if t['role'] == 'therapist' else 'Patient'}: {t['content']}"
        for t in turns
    )
    prompt = (
        "Transcript of a CBT therapy session:\n\n"
        f"{transcript_text}\n\n"
        "End the session (answer 'Yes') if EITHER condition holds; otherwise 'No'.\n\n"
        "A) Fully complete: the therapist has deeply explored the patient's concerns, applied a "
        "specific CBT technique, agreed a concrete homework/action plan (not vague), begun wrapping "
        "up, AND the patient has no concerns left. If still exploring, no technique/plan yet, or not "
        "wrapping up, A is not met.\n\n"
        "B) Genuine farewell: either the therapist OR the patient is actually saying goodbye to the "
        "other to end today's session (e.g. 'Take care, see you next week'). Do NOT count a goodbye "
        "that appears inside a story, quote, or memory the patient is recounting (e.g. 'she said "
        "goodbye and walked out') — that is narration, not closing. When unsure, treat it as an "
        "anecdote and do not end.\n\n"
        "When in doubt, answer 'No'. Answer with ONLY 'Yes' or 'No'."
    )
    async with SEM:
        resp = await client.chat.completions.create(
            model=MODERATOR_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=3,
            temperature=0.0,
            extra_body={"thinking": {"type": "disabled"}},
        )
    answer = resp.choices[0].message.content.strip().lower()
    return answer.startswith("yes")


# ── Main interview loop (batched) ─────────────────────────────────────────────

async def run_all_interviews(
    model, tokenizer, client: AsyncOpenAI, vignettes: list[dict], model_label: str,
) -> list[dict]:
    """
    Run all vignettes in lockstep with batched GPU inference.
    Each round: batched GPU therapist turns, then concurrent API patient + moderator.
    """
    n = len(vignettes)
    all_turns: list[list[dict]] = [[] for _ in range(n)]
    active = list(range(n))

    # ── Round 0: therapist opening ──
    print(f"  Round 0: therapist opening for {len(active)} vignettes (batch_size={THERAPIST_BATCH_SIZE})...", flush=True)
    append_batched_therapist_turns(model, tokenizer, all_turns, active)
    print(f"    done ({len(active)} openings)")

    # ── Rounds 1..MAX_TURNS-1: patient → therapist → moderator ──
    for round_num in range(1, MAX_TURNS):
        if not active:
            break

        # Patient turns — all concurrent
        print(f"  Round {round_num}: patient×{len(active)}...", end=" ", flush=True)
        patient_tasks = [
            patient_turn_async(client, vignettes[idx], all_turns[idx])
            for idx in active
        ]
        patient_responses = await asyncio.gather(*patient_tasks)
        for i, idx in enumerate(active):
            all_turns[idx].append({"role": "patient", "content": patient_responses[i]})

        # Therapist turns — batched on GPU
        print(f"therapist×{len(active)} (batch={THERAPIST_BATCH_SIZE})...", end=" ", flush=True)
        append_batched_therapist_turns(model, tokenizer, all_turns, active)

        # Moderator checks — concurrent for eligible conversations
        eligible = [
            idx for idx in active
            if (len(all_turns[idx]) + 1) // 2 >= MIN_TURNS
        ]
        ended = set()
        if eligible:
            print(f"moderator×{len(eligible)}...", end=" ", flush=True)
            mod_tasks = [
                should_end_session_async(client, all_turns[idx])
                for idx in eligible
            ]
            mod_results = await asyncio.gather(*mod_tasks)
            for i, idx in enumerate(eligible):
                if mod_results[i]:
                    ended.add(idx)

        active = [idx for idx in active if idx not in ended]
        print(f"ended={len(ended)}, active={len(active)}")

    # Build results
    results = []
    for idx in range(n):
        results.append({
            "model": model_label,
            "vignette": vignettes[idx]["name"],
            "turns": all_turns[idx],
        })
    return results


print("Batched async conversation engine defined.")

Batched async conversation engine defined.


## Load Model & Run Simulated Interviews

In [17]:
def load_therapist_model(model_path: str, base_model_name: str | None = None):
    """Load a therapist model for inference."""
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    path = Path(model_path)

    # Check for adapter_config.json at top level or in lora-adapters/ subfolder
    if (path / "adapter_config.json").exists():
        adapter_path = path
    elif (path / "lora-adapters" / "adapter_config.json").exists():
        adapter_path = path / "lora-adapters"
    else:
        adapter_path = None

    if adapter_path is not None:
        from peft import PeftModel
        assert base_model_name, "base_model_name required when loading a PEFT adapter"
        print(f"  Detected PEFT adapter at {adapter_path} — loading base + adapter and merging…")
        tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        base = AutoModelForCausalLM.from_pretrained(
            base_model_name, torch_dtype=dtype, device_map="auto", trust_remote_code=True
        )
        model = PeftModel.from_pretrained(base, str(adapter_path))
        model = model.merge_and_unload()
    else:
        print(f"  Loading full model from {model_path}…")
        tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=dtype, device_map="auto", trust_remote_code=True
        )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    total_params = sum(p.numel() for p in model.parameters()) / 1e9
    print(f"  Loaded {total_params:.1f}B parameters.")
    return model, tokenizer


print("load_therapist_model defined.")

load_therapist_model defined.


In [18]:
EVAL_JSONL.write_text("")

def save_transcripts(transcripts: list[dict]):
    """Append transcripts to JSONL."""
    with open(EVAL_JSONL, "a") as f:
        for t in transcripts:
            f.write(json.dumps(t) + "\n")

# ── Load and evaluate the single model ────────────────────────────────────────
print("=" * 60)
print(f"MODEL: {MODEL_NAME}")
print(f"PATH : {MODEL_PATH}")
print(f"Vignettes : {len(VIGNETTES)}")
print("=" * 60)

if MODEL_NAME == "base":
    model, tok = load_therapist_model(str(MODEL_PATH))
else:
    model, tok = load_therapist_model(str(MODEL_PATH), base_model_name=BASE_MODEL_NAME)

t0 = time.perf_counter()
transcripts = await run_all_interviews(
    model, tok, ds_async, VIGNETTES, MODEL_NAME
)
elapsed = time.perf_counter() - t0
save_transcripts(transcripts)
print(f"\nDone: {len(transcripts)} transcripts in {elapsed:.1f}s")

del model, tok
torch.cuda.empty_cache()

print(f"Transcripts → {EVAL_JSONL} ({EVAL_JSONL.stat().st_size / 1024:.1f} KB)")

MODEL: Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged
PATH : /data_1_8TB_ssd/kevint/models/Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged
Vignettes : 100
  Loading full model from /data_1_8TB_ssd/kevint/models/Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged…


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

  Loaded 4.2B parameters.
  Round 0: therapist opening for 100 vignettes (batch_size=8)...
    done (100 openings)
  Round 1: patient×100... therapist×100 (batch=8)... ended=0, active=100
  Round 2: patient×100... therapist×100 (batch=8)... ended=0, active=100
  Round 3: patient×100... therapist×100 (batch=8)... ended=0, active=100
  Round 4: patient×100... therapist×100 (batch=8)... moderator×100... ended=0, active=100
  Round 5: patient×100... therapist×100 (batch=8)... moderator×100... ended=0, active=100
  Round 6: patient×100... therapist×100 (batch=8)... moderator×100... ended=2, active=98
  Round 7: patient×98... therapist×98 (batch=8)... moderator×98... ended=4, active=94
  Round 8: patient×94... therapist×94 (batch=8)... moderator×94... ended=6, active=88
  Round 9: patient×88... therapist×88 (batch=8)... moderator×88... ended=13, active=75
  Round 10: patient×75... therapist×75 (batch=8)... moderator×75... ended=14, active=61
  Round 11: patient×61... therapist×61 (batch=8)..

## Async LLM-as-Judge: CTRS-R Scoring

One judge call per transcript scores all 11 CTRS-R items in a single structured JSON output.

In [19]:
def build_judge_prompt(vignette: dict, turns: list[dict]) -> str:
    """Build a CTRS-R judge prompt that scores all 11 items in one call."""
    transcript_text = "\n".join(
        f"{'Therapist' if t['role'] == 'therapist' else 'Patient'}: {t['content']}"
        for t in turns
    )

    rubric_text = ""
    for item in CTSR_ITEMS:
        anchors_text = "\n".join(f"      {k} = {v}" for k, v in item["anchors"].items())
        rubric_text += (
            f"  Item {item['number']}: {item['name']} (key: {item['key']})\n"
            f"    Criteria:\n{item['criteria']}\n"
            f"    Score anchors:\n{anchors_text}\n\n"
        )

    return (
        "You are a CBT clinical supervisor scoring a therapy session using the "
        "Cognitive Therapy Rating Scale – Revised (CTRS-R) by the Beck Institute.\n\n"
        "Scoring Key:\n"
        "0 = Item was NOT PRESENT.\n"
        "1 = Item was present but was UNSATISFACTORY.\n"
        "2 = Item was present and performed with MODERATE SKILL.\n"
        "3 = Item was present and performed VERY WELL.\n\n"
        f"Patient background:\n{vignette['background']}\n\n"
        f"Transcript:\n{transcript_text}\n\n"
        f"CTRS-R Rubric (11 items):\n{rubric_text}\n"
        "Score the therapist on ALL 11 items. For each item, provide your reasoning "
        "with specific transcript references, then assign the score.\n\n"
        "Output ONLY a JSON object with this exact structure:\n"
        "{\n"
        '  "item_1_agenda": {"reasoning": "...", "score": N},\n'
        '  "item_2_feedback": {"reasoning": "...", "score": N},\n'
        '  "item_3_understanding": {"reasoning": "...", "score": N},\n'
        '  "item_4_interpersonal_effectiveness": {"reasoning": "...", "score": N},\n'
        '  "item_5_collaboration": {"reasoning": "...", "score": N},\n'
        '  "item_6_pacing": {"reasoning": "...", "score": N},\n'
        '  "item_7_guided_discovery": {"reasoning": "...", "score": N},\n'
        '  "item_8_focus_cognitions_behaviors": {"reasoning": "...", "score": N},\n'
        '  "item_9_strategy_for_change": {"reasoning": "...", "score": N},\n'
        '  "item_10_application_cbt_technique": {"reasoning": "...", "score": N},\n'
        '  "item_11_action_plan": {"reasoning": "...", "score": N}\n'
        "}\n\n"
        "Rules:\n"
        "- Output ONLY the JSON object, no other text.\n"
        "- Each score must be an integer 0–3.\n"
        "- Each reasoning must be concise (2–4 sentences) with specific transcript references.\n"
        "- Do NOT use backslash-escaped single quotes in strings. Use plain apostrophes."
    )


def _sanitize_json(text: str) -> str:
    """Fix common JSON issues from LLM output (e.g. escaped single quotes)."""
    # Replace invalid \' (escaped single quote) with plain '
    text = text.replace("\\'", "'")
    return text


async def score_transcript_async(
    client: AsyncOpenAI, vignette: dict, turns: list[dict],
) -> dict:
    """Score all 11 CTRS-R items for a single transcript in one API call."""
    prompt = build_judge_prompt(vignette, turns)
    async with SEM:
        resp = await client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=JUDGE_MAX_TOKENS,
            temperature=0.0,
            extra_body={"thinking": {"type": "disabled"}},
        )
    raw = resp.choices[0].message.content.strip()

    cleaned = raw
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)

    cleaned = _sanitize_json(cleaned)

    results = {}
    try:
        data = json.loads(cleaned)
        for item in CTSR_ITEMS:
            key = item["key"]
            if key in data and isinstance(data[key], dict):
                score = int(data[key].get("score", 0))
                score = max(0, min(3, score))
                reasoning = data[key].get("reasoning", "")
            else:
                score = float("nan")
                reasoning = f"MISSING_KEY: {key}"
            results[key] = score
            results[key + "_rationale"] = reasoning
    except (json.JSONDecodeError, ValueError):
        # Fallback: try to extract scores with regex
        for item in CTSR_ITEMS:
            key = item["key"]
            pattern = rf'"{key}".*?"score"\s*:\s*(\d)'
            m = re.search(pattern, cleaned)
            if m:
                results[key] = max(0, min(3, int(m.group(1))))
                results[key + "_rationale"] = f"PARTIAL_PARSE: {cleaned[:200]}"
            else:
                results[key] = float("nan")
                results[key + "_rationale"] = f"PARSE_ERROR: {raw[:200]}"

    return results


async def score_all_transcripts(
    client: AsyncOpenAI, transcripts: list[dict], vignettes: list[dict],
) -> list[dict]:
    """
    Score all transcripts — one API call per transcript (all 11 items at once).
    Returns a list of row dicts ready for pd.DataFrame.
    """
    vignette_map = {v["name"]: v for v in vignettes}

    tasks = [
        score_transcript_async(client, vignette_map[t["vignette"]], t["turns"])
        for t in transcripts
    ]

    print(f"  Firing {len(tasks)} judge calls (semaphore={API_CONCURRENCY})...", flush=True)
    results = await asyncio.gather(*tasks)
    print(f"  All {len(tasks)} calls complete.")

    rows: list[dict] = []
    for t_idx, (transcript, result) in enumerate(zip(transcripts, results)):
        row = {"model": transcript["model"], "vignette": transcript["vignette"]}
        row.update(result)
        rows.append(row)

    return rows


print("Async scoring functions defined.")

Async scoring functions defined.


In [20]:
t0 = time.perf_counter()
scores_list = await score_all_transcripts(ds_async, transcripts, VIGNETTES)
elapsed = time.perf_counter() - t0

scores_df = pd.DataFrame(scores_list)
scores_df.to_csv(EVAL_CSV, index=False)

print(f"\nScoring complete in {elapsed:.1f}s")
print(f"Scores table: {scores_df.shape}")
print(f"Scores CSV → {EVAL_CSV} ({EVAL_CSV.stat().st_size / 1024:.1f} KB)")

# Show summary
print(f"\n{'=' * 60}")
print(f"CTRS-R Summary for: {MODEL_NAME}")
print(f"{'=' * 60}")
means = scores_df[ITEM_KEYS].mean()
for key, label in zip(ITEM_KEYS, ITEM_LABELS):
    print(f"  {label.replace(chr(10), ' '):30s}: {means[key]:.2f}")
print(f"  {'OVERALL MEAN':30s}: {means.mean():.2f}")
scores_df

  Firing 100 judge calls (semaphore=500)...
  All 100 calls complete.

Scoring complete in 11.5s
Scores table: (100, 24)
Scores CSV → ../reports/Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged/ctsr_eval_scores.csv (353.2 KB)

CTRS-R Summary for: Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged
  Agenda                        : 1.12
  Feedback                      : 1.37
  Understanding                 : 2.71
  Interpersonal Effect.         : 2.86
  Collaboration                 : 2.75
  Pacing                        : 2.61
  Guided Discovery              : 2.96
  Focus on Key Cog. & Beh.      : 2.98
  Strategy for Change           : 2.90
  Application of CBT Tech.      : 2.97
  Action Plan                   : 2.66
  OVERALL MEAN                  : 2.54


,model,vignette,item_1_agenda,item_1_agenda_rationale,item_2_feedback,item_2_feedback_rationale,item_3_understanding,item_3_understanding_rationale,item_4_interpersonal_effectiveness,item_4_interpersonal_effectiveness_rationale,...,item_7_guided_discovery,item_7_guided_discovery_rationale,item_8_focus_cognitions_behaviors,item_8_focus_cognitions_behaviors_rationale,item_9_strategy_for_change,item_9_strategy_for_change_rationale,item_10_application_cbt_technique,item_10_application_cbt_technique_rationale,item_11_action_plan,item_11_action_plan_rationale
0,Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged,cactus_00000,1,The therapist asked about the client's week an...,1,The therapist asked for feedback at the end ('...,3,The therapist consistently reflected and summa...,3,The therapist demonstrated concern ('I'm glad ...,...,3,The therapist used Socratic questioning to hel...,3,The therapist focused on the key cognition 'I ...,3,The therapist discussed a strategy of using gr...,3,The therapist skillfully guided the client thr...,2,The therapist reviewed the action plan (ground...
1,Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged,cactus_00001,2,The therapist provided a transition from the p...,1,The therapist asked for the client's input on ...,2,The therapist consistently reflected the clien...,3,"The therapist showed concern, provided positiv...",...,3,The therapist used Socratic questioning to hel...,3,The therapist focused on the specific cognitio...,3,The therapist discussed a strategy of examinin...,3,The therapist skillfully applied cognitive res...,3,The therapist developed a new Action Plan (rev...
2,Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged,cactus_00002,1,The therapist provided a transition from the p...,2,The therapist asked for the client's reaction ...,3,The therapist consistently listened and reflec...,3,"The therapist showed concern, provided positiv...",...,3,The therapist skillfully used Socratic questio...,3,The therapist focused on the most relevant cog...,3,The therapist discussed an overall strategy fo...,3,The therapist applied cognitive restructuring ...,3,The therapist reviewed the previous Action Pla...
3,Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged,cactus_00003,2,The therapist provided a transition from the p...,1,The therapist asked for the client's prioritie...,3,The therapist consistently listened and reflec...,3,"The therapist demonstrated concern, provided p...",...,3,The therapist used Socratic questioning to hel...,3,The therapist focused on specific cognitions (...,3,The therapist discussed an overall strategy (i...,3,The therapist skillfully applied cognitive res...,2,The therapist reviewed no previous Action Plan...
4,Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged,cactus_00004,1,The therapist provided a transition from the p...,1,The therapist asked for the client's reaction ...,3,The therapist consistently listened and reflec...,3,"The therapist demonstrated concern, provided p...",...,3,The therapist used Socratic questioning to hel...,3,The therapist focused on specific cognitions (...,3,The therapist discussed an overall strategy (c...,3,The therapist skillfully applied cognitive res...,3,The therapist reviewed the previous Action Pla...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged,cactus_00095,1,The therapist asked about the client's current...,1,The therapist did not explicitly ask for the c...,3,The therapist consistently listened and reflec...,3,"The therapist demonstrated concern, provided p...",...,3,The therapist used Socratic questioning to hel...,3,The therapist focused on specific cognitions (...,3,The therapist discussed an overall strategy (t...,3,The therapist skillfully applied a grounding t...,3,The therapist reviewed no previous Action Plan...
96,Qwen3.5-4B-SFT-KTO-CACTUS-iter_01-merged,cactus_00096,1,The therapist provided a transition from the p...,2,The therapist asked for the client's reaction ...,2,The therap